# G1 Academy Bonus - Task 9: low-level joint control + q/dq/kp/kd/tau visualizer

## Introduction
This task rebuilds the `rt/lowcmd` publisher lifecycle - message construction, CRC, freshness checking, and a single-joint `move_ll_joint` helper - then provides a small Jupyter UI (built with `ipywidgets` + `matplotlib`) so you can *feel* what `q`, `dq`, `kp`, `kd`, and `tau` each do: stream a commanded target continuously and plot commanded-vs-measured traces on demand.

`kp`/`kd` act as a virtual spring/damper pulling the joint toward the commanded `q`/`dq`; `tau` is a feed-forward torque added on top. High `kp` tracks position tighter but feels stiffer and can overshoot; `tau` alone with `kp=kd=0` is open-loop torque control with no position feedback at all.

In [1]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

_factory_config = None
def ensure_channel_factory(domain_id, interface):
    global _factory_config
    config = (int(domain_id), str(interface))
    if _factory_config is None:
        ChannelFactoryInitialize(*config)
        _factory_config = config
    elif _factory_config != config:
        raise RuntimeError(f"ChannelFactory already initialized as {_factory_config}; restart kernel for {config}.")
    return _factory_config

ensure_channel_factory(0, "eth0")

class Latest:
    def __init__(self, topic, message_type, queue_len=10):
        self.message = None
        self.timestamp = 0.0
        self.subscriber = ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self._callback, queue_len)
    def _callback(self, message):
        self.message = message
        self.timestamp = time.time()
    def fresh(self, max_age_s=0.5):
        return self.message is not None and time.time() - self.timestamp <= max_age_s

lowstate_sub = Latest("rt/lowstate", LowState_)

## Task 1 - Native `rt/lowcmd` publisher: message construction, CRC, and defaults
One `LowCmd_` message is reused across writes; every one of the 29 body joints gets a full motor command (`mode`, `q`, `dq`, `tau`, `kp`, `kd`), and the message CRC is recomputed on every write before publishing - an unset or stale CRC is silently rejected by the firmware.

In [2]:
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_
from unitree_sdk2py.utils.crc import CRC

LOWCMD_JOINTS = list(range(0, 29))  # left_leg 0-5, right_leg 6-11, waist 12-14, left_arm 15-21, right_arm 22-28
DEFAULT_KP = [60, 60, 60, 100, 40, 40, 60, 60, 60, 100, 40, 40, 60, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40]
DEFAULT_KD = [1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

class LowCmdPublisher:
    def __init__(self):
        self.crc = CRC()
        self.pub = ChannelPublisher("rt/lowcmd", LowCmd_); self.pub.Init()
        self.msg = unitree_hg_msg_dds__LowCmd_(); self.msg.mode_pr = 0
    def write(self, q, mode_machine, kp=None, kd=None, dq=0.0, tau=0.0):
        self.msg.mode_machine = int(mode_machine)
        kp = kp or DEFAULT_KP
        kd = kd or DEFAULT_KD
        for i in LOWCMD_JOINTS:
            cmd = self.msg.motor_cmd[i]
            cmd.mode = 1; cmd.q = float(q[i]); cmd.dq = float(dq); cmd.tau = float(tau)
            cmd.kp = float(kp[i]); cmd.kd = float(kd[i])
        self.msg.crc = self.crc.Crc(self.msg)
        self.pub.Write(self.msg)

lowcmd_pub = LowCmdPublisher()

def require_fresh(sub, name, max_age_s=0.5):
    """Reusable freshness policy: every write below must refuse to command joints
    from a stale or absent lowstate snapshot instead of holding the last-known q forever."""
    if not sub.fresh(max_age_s=max_age_s):
        raise RuntimeError(f"{name} is not fresh; refusing to command.")

def current_q_mode(timeout_s=3.0):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if lowstate_sub.fresh(max_age_s=1.0):
            msg = lowstate_sub.message
            return [float(msg.motor_state[i].q) for i in LOWCMD_JOINTS], int(msg.mode_machine)
        time.sleep(0.02)
    raise TimeoutError("Timed out waiting for fresh rt/lowstate.")

## Task 2 - `move_ll_joint(joint_id, q, dq, kp, kd, tau)`
Commands a single joint while holding every other joint at its currently observed position - the same pattern `sdk_wrapper.G1.move_ll_joint` uses for its `arm_sdk=False` path.

In [15]:
def move_ll_joint(joint_id, q, dq=0.0, kp=40.0, kd=1.0, tau=0.0):
    require_fresh(lowstate_sub, "rt/lowstate")
    q_all, mode_machine = current_q_mode()
    kp_all, kd_all = list(DEFAULT_KP), list(DEFAULT_KD)
    q_all[joint_id] = float(q); kp_all[joint_id] = float(kp); kd_all[joint_id] = float(kd)
    lowcmd_pub.write(q_all, mode_machine, kp=kp_all, kd=kd_all, dq=dq, tau=tau)

move_ll_joint(22, 1, dq=0.2, kp=40.0, kd=1.0)

## Task 3 - Jupyter UI: visualize q/dq/kp/kd/tau live
Sliders pick a joint and target `q`/`dq`/`kp`/`kd`/`tau`. "Stream command" starts a background thread that keeps calling `move_ll_joint` at a fixed rate with the current slider values (so you can tweak `kp`/`kd` live). "Capture & plot 3s" samples the *measured* `q`/`dq`/`tau_est` for the selected joint for three seconds and plots each against its commanded target - the gap between the dashed commanded line and the measured trace is exactly the tracking error `kp`/`kd` are fighting to close.

In [16]:
import threading
import ipywidgets as widgets
import matplotlib.pyplot as plt

joint_slider = widgets.IntSlider(min=0, max=28, value=22, description="joint_id")
q_slider = widgets.FloatSlider(min=-3.0, max=3.0, step=0.01, value=0.0, description="q target")
dq_slider = widgets.FloatSlider(min=-2.0, max=2.0, step=0.01, value=0.0, description="dq target")
kp_slider = widgets.FloatSlider(min=0.0, max=150.0, step=1.0, value=40.0, description="kp")
kd_slider = widgets.FloatSlider(min=0.0, max=5.0, step=0.05, value=1.0, description="kd")
tau_slider = widgets.FloatSlider(min=-5.0, max=5.0, step=0.05, value=0.0, description="tau (ff)")
stream_toggle = widgets.ToggleButton(value=False, description="Stream command")
capture_button = widgets.Button(description="Capture & plot 3s")
out = widgets.Output()

_stream_thread = None
_stream_stop = threading.Event()

def _stream_worker(rate_hz=50.0):
    while not _stream_stop.is_set():
        try:
            move_ll_joint(joint_slider.value, q_slider.value, dq_slider.value, kp_slider.value, kd_slider.value, tau_slider.value)
        except Exception:
            pass
        _stream_stop.wait(1.0 / rate_hz)

def _on_stream_toggle(change):
    global _stream_thread
    if change["new"]:
        _stream_stop.clear()
        _stream_thread = threading.Thread(target=_stream_worker, daemon=True)
        _stream_thread.start()
    else:
        _stream_stop.set()
        if _stream_thread is not None:
            _stream_thread.join(timeout=1.0)
stream_toggle.observe(_on_stream_toggle, names="value")

def _on_capture(_button):
    with out:
        out.clear_output(wait=True)
        joint_id = joint_slider.value
        target_q, target_dq, target_tau = q_slider.value, dq_slider.value, tau_slider.value
        samples_t, samples_q, samples_dq, samples_tau = [], [], [], []
        start = time.time()
        while time.time() - start < 3.0:
            if lowstate_sub.fresh():
                motor = lowstate_sub.message.motor_state[joint_id]
                samples_t.append(time.time() - start)
                samples_q.append(motor.q); samples_dq.append(motor.dq); samples_tau.append(motor.tau_est)
            time.sleep(0.02)
        fig, axes = plt.subplots(3, 1, figsize=(6, 6), sharex=True)
        axes[0].plot(samples_t, samples_q, label="measured q")
        axes[0].axhline(target_q, color="r", ls="--", label="commanded q")
        axes[0].set_ylabel("q (rad)"); axes[0].legend()
        axes[1].plot(samples_t, samples_dq, label="measured dq")
        axes[1].axhline(target_dq, color="r", ls="--", label="commanded dq")
        axes[1].set_ylabel("dq (rad/s)"); axes[1].legend()
        axes[2].plot(samples_t, samples_tau, label="measured tau_est")
        axes[2].axhline(target_tau, color="r", ls="--", label="commanded tau (ff)")
        axes[2].set_ylabel("tau (N.m)"); axes[2].set_xlabel("s"); axes[2].legend()
        fig.suptitle(f"joint {joint_id}  kp={kp_slider.value} kd={kd_slider.value}")
        plt.show()
capture_button.on_click(_on_capture)

ui = widgets.VBox([
    joint_slider, q_slider, dq_slider, kp_slider, kd_slider, tau_slider,
    widgets.HBox([stream_toggle, capture_button]), out,
])
display(ui)

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.